In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegressionCV

from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement

sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data


import matplotlib.pyplot as plt

## Load in the training data

In [ ]:
fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,labels = load_data(fnm,fBounds=(1,56),feature_list=['power'])
myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])


### Subselect prefrontal cortex power only

In [ ]:
pf = labels['powerFeatures']
idxs = np.zeros(len(pf))
for i in range(len(pf)):
    if pf[i][:2] == 'IL':
        idxs[i] = 1
power = power[:,idxs==1]
print(power.shape)

## Subselect only the windows we care about

In [ ]:
N = len(mouse)

indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

y = np.zeros(N)
y[indx_pos] = 1

mouse = mouse[indx_tot]
group = group[indx_tot]
expDate = expDate[indx_tot]
behavior = behavior[indx_tot]
behaviornon1 = behaviornon1[indx_tot]
time = time[indx_tot]
condition = condition[indx_tot]
y = y[indx_tot]

N = len(mouse)

training_set_idx = np.ones(N)
training_set_idx[mouse=='Mouse048'] = 0
training_set_idx[mouse=='Mouse7980'] = 0
training_set_idx[mouse=='Mouse7998'] = 0

### Still maintain the preprocessing method
power = power*10
power[power>6] = 6

X = power
X = X[indx_tot]

X_train = X[training_set_idx==1]
m_train = mouse[training_set_idx==1]
y_train = y[training_set_idx==1]

X_test = X[training_set_idx==0]
m_test = mouse[training_set_idx==0]
y_test = y[training_set_idx==0]


## Test set data

In [ ]:
fName = '/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_all_validate3.mat'
power_new,labels_new = load_data(fName,fBounds=(1,56),feature_list=['power'])
power_new = 10*power_new
power_new = power_new.astype(np.float32)
power_new[power_new>6] = 6

#Subselect prefrontal again
power_new = power_new[:,idxs==1]


X_new = power_new
windows_new = labels_new['windows']
mouse_new = np.squeeze(windows_new['mouse'])
expDate_new = np.squeeze(windows_new['expDate'])
group_new = np.squeeze(windows_new['group'])
condition_new = np.squeeze(windows_new['condition'])
behavior_new = np.squeeze(windows_new['behavior'])
time_new = np.squeeze(windows_new['time'])

#### Subselect new data

In [ ]:
idx_pos_new = (condition_new==4)&(behavior_new==1)
indx_neg_new = (behavior_new==2)&((condition_new==4)|(condition_new==6)|(condition_new==8))
y_new = np.zeros(len(mouse_new))
y_new[idx_pos_new] = 1
idx_tot_new = idx_pos_new|indx_neg_new

X_new = X_new[idx_tot_new]
mouse_new = mouse_new[idx_tot_new]
y_new = y_new[idx_tot_new]

#### Divide into training and testing set

In [ ]:
mice_new = np.unique(mouse_new)
nMice = len(mice_new)
mice_new_train = mice_new[:4]

ids = np.zeros(len(mouse_new))
for i in range(4):
    ids[mouse_new==mice_new_train[i]] = 1
X_train_new = X_new[ids==1,:]
X_test_new = X_new[ids==0,:]
y_train_new = y_new[ids==1]
y_test_new = y_new[ids==0]

In [ ]:
mice_new

#### Combine the training sets

In [ ]:
X_train_tot = np.vstack((X_train,X_train_new))
y_train_tot = np.concatenate((y_train,y_train_new))

## Learn the model

In [ ]:
model = LogisticRegressionCV(penalty='elasticnet',n_jobs=8,l1_ratios=[0.5,0.8,0.9,0.95,0.99],random_state=42,
                                solver='saga',max_iter=10000)
model.fit(X_train_tot,y_train_tot)

In [ ]:
myDict = {'model':model}

In [ ]:
coef_ = np.squeeze(model.coef_)
plt.plot(coef_)

### Now predict on original test set

In [ ]:
S_test = model.decision_function(X_test)
S_test_new = model.decision_function(X_test_new)

myDict['auc_test_total'] = roc_auc_score(y_test,S_test)
myDict['auc_test_new_total'] = roc_auc_score(y_test_new,S_test_new)

In [ ]:
mice_test = np.unique(m_test)
idx1 = mice_test[0]==m_test
idx2 = mice_test[1]==m_test
idx3 = mice_test[2]==m_test
myDict['mouse_0_auc'] = roc_auc_score(y_test[idx1],S_test[idx1])
myDict['mouse_1_auc'] = roc_auc_score(y_test[idx2],S_test[idx2])
myDict['mouse_2_auc'] = roc_auc_score(y_test[idx3],S_test[idx3])


In [ ]:
mice_test

In [ ]:
print(myDict['mouse_0_auc'],myDict['mouse_1_auc'],myDict['mouse_2_auc'])

In [ ]:
pickle.dump(myDict,open('ElasticNetResults.p','wb'))

## OK well we are going to switch to an L2 loss so I don't feel like I made a big mistake

In [ ]:
model2 = LogisticRegressionCV(penalty='l2',n_jobs=8,solver='saga',max_iter=10000,random_state=42)
model2.fit(X_train_tot,y_train_tot)

In [ ]:
coef_ = np.squeeze(model2.coef_)
plt.plot(coef_)

In [ ]:
S_test = model2.decision_function(X_test)
S_test_new = model2.decision_function(X_test_new)
myDict2 = {'model':model2}
myDict2['auc_test_total'] = roc_auc_score(y_test,S_test)
myDict2['auc_test_new_total'] = roc_auc_score(y_test_new,S_test_new)

mice_test = np.unique(m_test)
idx1 = mice_test[0]==m_test
idx2 = mice_test[1]==m_test
idx3 = mice_test[2]==m_test
myDict2['mouse_0_auc'] = roc_auc_score(y_test[idx1],S_test[idx1])
myDict2['mouse_1_auc'] = roc_auc_score(y_test[idx2],S_test[idx2])
myDict2['mouse_2_auc'] = roc_auc_score(y_test[idx3],S_test[idx3])


In [ ]:
print(myDict2['mouse_0_auc'],myDict2['mouse_1_auc'],myDict2['mouse_2_auc'])

In [ ]:
print(myDict2['auc_test_total'],myDict2['auc_test_new_total'])

In [ ]:
S_test_new = model2.decision_function(X_test_new)


In [ ]:
mice_new = np.unique(mouse_new)
nMice = len(mice_new)
mice_new_train = mice_new[:4]

ids = np.zeros(len(mouse_new))
for i in range(4):
    ids[mouse_new==mice_new_train[i]] = 1
X_train_new = X_new[ids==1,:]
X_test_new = X_new[ids==0,:]
y_train_new = y_new[ids==1]
y_test_new = y_new[ids==0]

In [ ]:
m_test_new = mouse_new[ids==0]

In [ ]:
mice_test = np.unique(m_test_new)
for i in range(4):
    idxs = m_test_new==mice_test[i]
    myDict2[mice_test[i]+'_auc'] = roc_auc_score(y_test_new[idxs],S_test_new[idxs])
    print(mice_test[i],myDict2[mice_test[i]+'_auc'])

In [ ]:
pickle.dump(myDict2,open('L2Results.p','wb'))